# 01 — Prepare Dataset items_prompts_tv_1

**Mục đích:** Build prompt/completion từ `SeanSunny/items_tv_v6` và push lên HuggingFace  
thành `SeanSunny/items_prompts_tv_1` (3 splits: train/val/test).

**Không cần GPU.**

**Yêu cầu:** `HF_TOKEN` có write access vào HuggingFace.

**Output:** `SeanSunny/items_prompts_tv_1` trên HF, `dataset_stats.md`

In [ ]:
# On Colab: uncomment and run this cell first
# !pip install datasets huggingface_hub --quiet

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset, DatasetDict

# ---- Config ----
SOURCE_DATASET = "SeanSunny/items_tv_v6"
OUTPUT_DATASET = "SeanSunny/items_prompts_tv_1"
SEED = 42

# HF_TOKEN: set env variable or paste below
HF_TOKEN = os.environ.get("HF_TOKEN", "")
# HF_TOKEN = "hf_xxx"  # uncomment if not in env

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not set. Export it or set HF_TOKEN above.")

random.seed(SEED)
np.random.seed(SEED)
print("Config OK")

In [ ]:
# Load source dataset
ds = load_dataset(SOURCE_DATASET)
print(ds)

splits = list(ds.keys())
val_key = "val" if "val" in splits else "validation"
print(f"\nAvailable splits: {splits}")
print(f"Using val split  : '{val_key}'")
print(f"Columns          : {ds['train'].column_names}")

In [ ]:
# Prompt builder — canonical definition (same as utils/prompt_builder.py)
PROMPT_TEMPLATE = """Sản phẩm này có giá bao nhiêu ?
Tiêu đề: {title}
Danh mục: {category}
Thương hiệu: {brand}
Mô tả: {description}
Thông số: {features}

Giá là: """


def build_prompt(item: dict) -> str:
    return PROMPT_TEMPLATE.format(
        title=item.get("title") or "",
        category=item.get("category") or "",
        brand=item.get("brand") or "Không rõ",
        description=item.get("description") or "Không có mô tả",
        features=item.get("features") or "Không có thông số",
    )


def build_example(item: dict, for_test: bool = False) -> dict:
    price = float(item["price"])
    return {
        "prompt": build_prompt(item),
        "completion": str(int(round(price))) if for_test else str(int(round(price / 1000))),
        "price_vnd_true": int(round(price)),
        "category": item.get("category") or "",
        "id": int(item.get("id") or 0),
    }


# Sanity check on one item
item0 = dict(ds["train"][0])
ex0 = build_example(item0, for_test=False)
print("=== Sample train example ===")
print(ex0["prompt"])
print(f"completion     : {ex0['completion']}")
print(f"price_vnd_true : {ex0['price_vnd_true']:,}")

In [ ]:
# Apply to all splits
def process_split(split, for_test=False):
    return split.map(
        lambda item: build_example(item, for_test=for_test),
        remove_columns=split.column_names,
        desc=f"Building prompts (test={for_test})",
        num_proc=1,
    )

train_prompts = process_split(ds["train"], for_test=False)
val_prompts = process_split(ds[val_key], for_test=False)
test_prompts = process_split(ds["test"], for_test=True)

out_ds = DatasetDict({
    "train": train_prompts,
    "val": val_prompts,
    "test": test_prompts,
})

print(out_ds)
print(f"\nSchema: {out_ds['train'].column_names}")

In [ ]:
# Dataset statistics
for split_name, split_data in out_ds.items():
    df = split_data.to_pandas()
    print(f"\n=== {split_name} ({len(df):,} items) ===")
    prices = df["price_vnd_true"]
    print(f"  Price (VND): mean={prices.mean():.0f}, median={prices.median():.0f}, "
          f"min={prices.min():,}, max={prices.max():,}")
    print(f"  Completion sample: {df['completion'].sample(5, random_state=42).tolist()}")
    print(f"  Category counts:")
    for cat, cnt in df["category"].value_counts().items():
        print(f"    {cat}: {cnt:,}")

In [ ]:
# Show 5 complete examples from train
df_train = out_ds["train"].to_pandas()
sample5 = df_train.sample(5, random_state=42)

for _, row in sample5.iterrows():
    print("=" * 70)
    print(row["prompt"])
    print(f"[COMPLETION: {row['completion']}  |  PRICE_VND: {row['price_vnd_true']:,}]")
    print()

In [ ]:
# Push to HuggingFace Hub
out_ds.push_to_hub(OUTPUT_DATASET, token=HF_TOKEN)
print(f"\nSuccessfully pushed {OUTPUT_DATASET} to HuggingFace Hub")

In [ ]:
# Save dataset_stats.md
lines = ["# Dataset Stats — items_prompts_tv_1\n\n"]

for split_name in ["train", "val", "test"]:
    df = out_ds[split_name].to_pandas()
    prices = df["price_vnd_true"]
    lines.append(f"## {split_name} ({len(df):,} items)\n\n")
    lines.append(f"- Price (VND): mean={prices.mean():.0f}, median={prices.median():.0f}, "
                 f"min={prices.min():,}, max={prices.max():,}\n")
    lines.append(f"- Categories ({df['category'].nunique()} unique):\n")
    for cat, cnt in df["category"].value_counts().items():
        lines.append(f"  - {cat}: {cnt:,}\n")
    lines.append("\n")

lines.append("## Sample 5 Prompts (train)\n\n")
for _, row in df_train.sample(5, random_state=42).iterrows():
    lines.append("```\n")
    lines.append(row["prompt"].rstrip() + "\n")
    lines.append(f"[COMPLETION: {row['completion']}  |  PRICE_VND: {row['price_vnd_true']:,}]\n")
    lines.append("```\n\n")

with open("dataset_stats.md", "w", encoding="utf-8") as f:
    f.writelines(lines)

print("Saved dataset_stats.md")

## Ket qua Phase 0 Notebook 2

**DONE:** Dataset `SeanSunny/items_prompts_tv_1` da push len HuggingFace.  
Share `dataset_stats.md` voi Claude de confirm truoc khi sang Phase 1.

**Buoc tiep:**
1. Share `profile_results.json` (tu notebook 00) voi Claude
2. Share `dataset_stats.md` voi Claude
3. Claude confirm `max_seq_length` va `max_new_tokens`
4. Bat dau Phase 1: `02_baseline_v0.ipynb` (can GPU)